# Machine Learning with Smile

[Smile](https://haifengl.github.io/) (Statistical Machine Intelligence and Learning Engine) is a machine
learning library for the JVM, with a Scala API. This notebook walks through a small workflow with it:
loading data, training and evaluating a classifier, fitting a regression, clustering, and plotting the
result with [plotly-scala](https://github.com/alexarchambault/plotly-scala).

It is adapted from the Scala notebooks that ship with Smile
([`shell/src/universal/notebooks`](https://github.com/haifengl/smile/tree/v3.1.1/shell/src/universal/notebooks)
in the [haifengl/smile](https://github.com/haifengl/smile) repository, by Haifeng Li), and uses two small
datasets copied from Smile's `data` directory into [`data/`](data/):
[`iris.arff`](data/iris.arff) (Fisher's iris data) and [`longley.arff`](data/longley.arff) (Longley's
macroeconomic data, from StatLib). Smile is licensed under the
[GNU GPL v3](https://github.com/haifengl/smile/blob/v3.1.1/LICENSE).

We use Smile 3.1.1, the latest release that runs on Java 17, which is the JVM of the
`almondsh/almond:0.15.0` image. Smile 4 and later need Java 21 or newer.

In [1]:
import $ivy.`com.github.haifengl::smile-scala:3.1.1`
import $ivy.`org.plotly-scala::plotly-almond:0.8.5`
// Smile logs with SLF4J, this silences it
import $ivy.`org.slf4j:slf4j-nop:2.0.12`

import $ivy.$
import $ivy.$
import $ivy.$

In [2]:
import smile.read
import smile.classification.{cart, randomForest}
import smile.regression.{lm, ridge}
import smile.clustering.kmeans
import smile.feature.extraction.PCA
import smile.data.DataFrame
import smile.data.formula._
import smile.math.MathEx
import smile.validation._
import smile.validation.metric._

// restrict the output height to avoid scrolling in output cells
repl.pprinter() = repl.pprinter().copy(defaultHeight = 5)

import smile.read
import smile.classification.{cart, randomForest}
import smile.regression.{lm, ridge}
import smile.clustering.kmeans
import smile.feature.extraction.PCA
import smile.data.DataFrame
import smile.data.formula._
import smile.math.MathEx
import smile.validation._
import smile.validation.metric._

## Loading data

`smile.read` reads CSV, ARFF, JSON, Parquet, Arrow, and other formats into a `DataFrame`.

In [3]:
val iris = read.arff("data/iris.arff")

iris: DataFrame = [sepallength: float, sepalwidth: float, petallength: float, petalwidth: float, class: byte nominal[Iris-setosa, Iris-versicolor, Iris-virginica]]
+-----------+----------+-----------+----------+-----------+
|sepallength|sepalwidth|petallength|petalwidth|      class|
...

In [4]:
println(iris.summary())

[column: String, count: long, min: double, avg: double, max: double]
+-----------+-----+---+--------+---+
|     column|count|min|     avg|max|
+-----------+-----+---+--------+---+
|sepallength|  150|4.3|5.843333|7.9|
| sepalwidth|  150|  2|   3.054|4.4|
|petallength|  150|  1|3.758667|6.9|
| petalwidth|  150|0.1|1.198667|2.5|
+-----------+-----+---+--------+---+



A `Formula` says which column is the response and which are the predictors. `"class" ~ "."` predicts
`class` from all the other columns.

In [5]:
val formula: Formula = "class" ~ "."

formula: Formula = class ~ .

## Classification

Let's split the data into a training and a test set, with a fixed seed so that the results are
reproducible.

In [6]:
MathEx.setSeed(19650218)

val permutation = MathEx.permutate(iris.size)
val train = iris.of(permutation.take(100): _*)
val test = iris.of(permutation.drop(100): _*)

permutation: Array[Int] = Array(
  107,
  76,
  43,
...
train: DataFrame = [sepallength: float, sepalwidth: float, petallength: float, petalwidth: float, class: byte nominal[Iris-setosa, Iris-versicolor, Iris-virginica]]
+-----------+----------+-----------+----------+---------------+
|sepallength|sepalwidth|petallength|petalwidth|          class|
...
test: DataFrame = [sepallength: float, sepalwidth: float, petallength: float, petalwidth: float, class: byte nominal[Iris-setosa, Iris-versicolor, Iris-virginica]]
+-----------+----------+-----------+----------+---------------+
|sepallength|sepalwidth|petallength|petalwidth|          class|
...

We train a random forest on the training set, and look at its accuracy and confusion matrix on the
test set.

In [7]:
val forest = randomForest(formula, train, ntrees = 100)

val truth = formula.y(test).toIntArray
val predictions = forest.predict(test)

Accuracy.of(truth, predictions)

forest: smile.classification.RandomForest = smile.classification.RandomForest@4d00fd3
truth: Array[Int] = Array(
  1,
  1,
  1,
...
predictions: Array[Int] = Array(
  1,
  1,
  1,
...
res7_3: Double = 0.92

In [8]:
println(ConfusionMatrix.of(truth, predictions))

ROW=truth and COL=predicted
class  0 |      19 |       0 |       0 |
class  1 |       0 |      18 |       2 |
class  2 |       0 |       2 |       9 |


Random forests also tell how much each variable contributes to the predictions:

In [9]:
formula.x(train).names().zip(forest.importance()).sortBy(-_._2)

res9: Array[(String, Double)] = Array(
  ("petalwidth", 49.349462683980676),
  ("petallength", 41.961613613039376),
  ("sepallength", 17.166145489291626),
...

With a dataset this small, cross-validation gives a more reliable estimate than a single split.
`cv.classification` trains and evaluates a model on each of `k` folds. Here we compare a single
decision tree (CART) and the random forest.

In [10]:
println(cv.classification(10, formula, iris) { (f, data) => cart(f, data) })

{
  fit time: 1.922 ms ± 0.799,
  score time: 0.025 ms ± 0.010,
  validation data size: 15 ± 0,
  error: 1 ± 1,
  accuracy: 92.67% ± 6.63,
  cross entropy: 0.2364 ± 0.1635
}


In [11]:
println(cv.classification(10, formula, iris) { (f, data) => randomForest(f, data, ntrees = 100) })

{
  fit time: 9.336 ms ± 2.411,
  score time: 0.523 ms ± 0.076,
  validation data size: 15 ± 0,
  error: 1 ± 1,
  accuracy: 96.00% ± 3.44,
  cross entropy: 0.1813 ± 0.0738
}


## Regression

The Longley dataset has a few macroeconomic variables of the US, for 16 years. We fit an ordinary least
squares model predicting the number of people employed from the other variables. Printing the model
shows the coefficients, their significance, and goodness-of-fit statistics.

In [12]:
val longley = read.arff("data/longley.arff")

longley: DataFrame = [deflator: float, GNP: float, unemployed: int, armed_forces: int, population: int, year: int, employed: int]
+--------+------+----------+------------+----------+----+--------+
|deflator|   GNP|unemployed|armed_forces|population|year|employed|
...

In [13]:
println(lm("employed" ~ ".", longley))

Linear Model:

Residuals:
       Min          1Q      Median          3Q         Max
 -410.1149   -155.5501    -17.2687    267.3399    455.3940

Coefficients:
                  Estimate Std. Error    t value   Pr(>|t|)
Intercept       -3482258.1299 890420.3117    -3.9108     0.0036 **
deflator           15.0617    84.9149     0.1774     0.8631 
GNP                -0.0358     0.0335    -1.0695     0.3127 
unemployed         -2.0202     0.4884    -4.1364     0.0025 **
armed_forces       -1.0332     0.2143    -4.8220     0.0009 ***
population         -0.0511     0.2261    -0.2261     0.8262 
year             1829.1512   455.4785     4.0159     0.0030 **
---------------------------------------------------------------------
Significance codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 304.8541 on 9 degrees of freedom
Multiple R-squared: 0.9955,    Adjusted R-squared: 0.9925
F-statistic: 330.2853 on 7 and 9 DF,  p-value: 4.984e-10



The predictors of this dataset are highly correlated, which makes ordinary least squares unstable. Ridge
regression shrinks the coefficients to address that. Let's compare the two with leave-one-out
cross-validation.

In [14]:
println(loocv.regression("employed" ~ ".", longley) { (f, data) => lm(f, data) })

{
  fit time: 0.273 ms,
  score time: 0.019 ms,
  validation data size:: 16,
  RSS: 2886891.6146,
  MSE: 180430.7259,
  RMSE: 424.7714,
  MAD: 333.4258,
  R2: 98.44%
}


In [15]:
println(loocv.regression("employed" ~ ".", longley) { (f, data) => ridge(f, data, 0.0057) })

{
  fit time: 0.326 ms,
  score time: 0.012 ms,
  validation data size:: 16,
  RSS: 2639260.4187,
  MSE: 164953.7762,
  RMSE: 406.1450,
  MAD: 313.3175,
  R2: 98.57%
}


## Clustering

Without looking at the labels, can we recover the iris species? We run k-means with 3 clusters on the
four measurements, and compare the clusters with the actual species with the adjusted Rand index (1
means a perfect match, 0 what one would expect by chance).

In [16]:
val x = formula.x(iris).toArray()
val y = formula.y(iris).toIntArray

MathEx.setSeed(19650218)
val clusters = kmeans(x, k = 3, runs = 10)

x: Array[Array[Double]] = Array(
  Array(5.099999904632568, 3.5, 1.399999976158142, 0.20000000298023224),
  Array(4.900000095367432, 3.0, 1.399999976158142, 0.20000000298023224),
  Array(
...
y: Array[Int] = Array(
  0,
  0,
  0,
...
clusters: smile.clustering.KMeans = Cluster distortion: 78.94084
Cluster size of 150 data points:
Cluster    1     50 (33.3%)
Cluster    2     62 (41.3%)
Cluster    3     38 (25.3%)


In [17]:
AdjustedRandIndex.of(y, clusters.y)

res17: Double = 0.7302382722834697

## Plotting

To see the clusters, we project the data on its first two principal components, and plot it with
plotly-scala, with one trace per cluster.

In [18]:
val pca = PCA.fit(x).getProjection(2)
println(pca.varianceProportion().take(2).map(p => f"$p%.3f").mkString("Variance explained by PC1, PC2: ", ", ", ""))

val projected = pca(x)

Variance explained by PC1, PC2: 0.925, 0.053


pca: PCA = smile.feature.extraction.PCA@290a499e
projected: Array[Array[Double]] = Array(
  Array(-2.684207165223753, -0.3266072408095235),
  Array(-2.715390583822476, 0.1695568033534327),
  Array(-2.8898196363071396, 0.13734570694237025),
...

In [19]:
import plotly._, plotly.element._, plotly.layout._, plotly.Almond._

val species = iris.column("class").toStringArray

val traces = (0 until clusters.k).map { c =>
  val idx = clusters.y.indices.filter(clusters.y(_) == c)
  Scatter(idx.map(projected(_)(0)), idx.map(projected(_)(1)))
    .withMode(ScatterMode(ScatterMode.Markers))
    .withName(s"Cluster ${c + 1}")
    .withText(idx.map(species(_)))
}

val layout = Layout()
  .withTitle("k-means clusters of the iris data")
  .withXaxis(Axis().withTitle("PC1"))
  .withYaxis(Axis().withTitle("PC2"))

plot(traces, layout)

import plotly._, plotly.element._, plotly.layout._, plotly.Almond._


species: Array[String] = Array(
  "Iris-setosa",
  "Iris-setosa",
  "Iris-setosa",
...
traces: IndexedSeq[Scatter] = Vector(
  Scatter(
    x = Some(
      value = Doubles(
...
layout: Layout = Layout(
  title = Some(value = "k-means clusters of the iris data"),
  legend = None,
  width = None,
...
res19_4: String = "plot-2d13d00d-8324-411a-8790-fe502ba2f9d2"